## 🎯 Learning Objectives
* Understand the limitations of single-head attention and the motivation for multi-head attention.
* Grasp the mechanics of multi-head attention, including parallel attention heads and concatenation.
* Comprehend why positional encoding is crucial for Transformers and how it's implemented.
* Implement basic multi-head attention and positional encoding layers in PyTorch.
* Analyze the impact and interpret the output of these components within a Transformer block.


## Multi-Head Attention and Positional Encoding: Giving Transformers Context and Order

Welcome to Lesson DL02-L10! In the previous lessons, we explored the foundational concept of attention, particularly scaled dot-product attention. While powerful, a single attention mechanism has its limitations. Today, we'll dive into two crucial enhancements that make the Transformer architecture so effective: **Multi-Head Attention** and **Positional Encoding**.

### The Power of Multiple Perspectives: Multi-Head Attention

Imagine you're trying to understand a complex piece of art. One person might focus on the colors, another on the brushstrokes, a third on the historical context, and a fourth on the emotional impact. Each perspective offers a unique lens, and combining them provides a much richer, more nuanced understanding than any single view alone.

This is precisely the intuition behind **Multi-Head Attention**. Instead of performing a single attention function, the input queries, keys, and values are linearly projected *h* different times with different, learned linear projections. Each of these projected sets then undergoes an independent scaled dot-product attention operation. These *h* parallel attention layers are called "heads."

**Why multiple heads?**

1.  **Diverse Relationships:** Each head can learn to attend to different parts of the input sequence or different types of relationships. For example, one head might focus on syntactic dependencies (e.g., subject-verb agreement), while another might capture semantic relationships (e.g., synonyms or related concepts). This allows the model to capture a richer set of contextual information.
2.  **Increased Representational Capacity:** By having multiple 


representation subspaces,


the model can attend to information from different positions and with different meanings simultaneously.

**How it works (Step-by-Step):**

1.  **Linear Projections:** The input Query (Q), Key (K), and Value (V) matrices are each projected `h` times into different lower-dimensional spaces. For example, if `d_model` is the embedding dimension and `h` is the number of heads, each head will work with `d_k = d_model / h` dimensions.
2.  **Parallel Attention:** For each of the `h` heads, a scaled dot-product attention is computed independently using its projected `Q_i`, `K_i`, `V_i`.
3.  **Concatenation:** The output values from all `h` attention heads are concatenated back together along the last dimension.
4.  **Final Linear Projection:** The concatenated output is then passed through a final linear layer to project it back to the original `d_model` dimension. This allows the model to combine the information learned by all heads into a single, coherent representation.

### Remembering the Order: Positional Encoding

Transformers, unlike Recurrent Neural Networks (RNNs) or Convolutional Neural Networks (CNNs), process all tokens in a sequence simultaneously. This parallel processing is a huge advantage for speed, but it comes with a critical drawback: the model has no inherent understanding of the *order* of tokens in the sequence. Without positional information, "The dog bit the man" would be indistinguishable from "The man bit the dog" in terms of token identity alone.

**Positional Encoding** solves this problem by injecting information about the relative or absolute position of tokens into the input embeddings. This is done by adding a unique vector to each token's embedding based on its position in the sequence.

**How it works (Sinusoidal Positional Encoding):**

The original Transformer paper proposed using fixed, non-learnable sinusoidal functions for positional encoding. These functions have a few key advantages:

1.  **Uniqueness:** Each position gets a unique encoding.
2.  **Relative Position Information:** The sinusoidal functions allow the model to easily learn to attend to relative positions (e.g., "the word two positions before me"). This is because `sin(x + k)` can be expressed as a linear function of `sin(x)` and `cos(x)`.
3.  **Generalization:** They can generalize to sequence lengths longer than those seen during training, as the functions are continuous.

The positional encoding for a given position `pos` and dimension `i` within the `d_model` embedding is calculated as:

*   `PE(pos, 2i) = sin(pos / 10000^(2i / d_model))`
*   `PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))`

Here, `pos` is the token's position in the sequence (0, 1, 2, ...), `i` is the dimension index (0, 1, 2, ..., `d_model`/2 - 1), and `d_model` is the embedding dimension. By alternating sine and cosine functions, and varying the frequency across dimensions, a rich and distinct positional signal is created for each token.

These two mechanisms – Multi-Head Attention for diverse contextual understanding and Positional Encoding for sequence order – are fundamental to the Transformer's ability to process complex sequential data like natural language.


In [ ]:
import torch
import torch.nn as nn
import math

# Set a random seed for reproducibility
torch.manual_seed(42)

class PositionalEncoding(nn.Module):
    """Injects positional information into the input embeddings using sinusoidal functions."""
    def __init__(self, d_model: int, max_len: int = 5000, dropout_rate: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout_rate)

        # Create a positional encoding matrix of shape (max_len, d_model)
        # This matrix will store the sine/cosine values for each position and dimension
        pe = torch.zeros(max_len, d_model)

        # Create a tensor for positions (0, 1, ..., max_len-1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Calculate the 'division term' for the sinusoidal functions
        # This term ensures that different dimensions have different wavelengths
        # 10000^(2i/d_model) where i goes from 0 to d_model/2 - 1
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Apply sine to even indices in the d_model dimension
        pe[:, 0::2] = torch.sin(position * div_term)
        # Apply cosine to odd indices in the d_model dimension
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add an extra dimension for batch size (1, max_len, d_model)
        # This allows broadcasting when adding to input embeddings
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor (batch_size, seq_len, d_model)
        Returns:
            Tensor with positional encoding added (batch_size, seq_len, d_model)
        """
        # Add positional encoding to the input embeddings
        # x.size(1) is the current sequence length
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class MultiHeadAttention(nn.Module):
    """Implements a simplified Multi-Head Attention mechanism."""
    def __init__(self, d_model: int, num_heads: int, dropout_rate: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads # Dimension of K, Q, V for each head

        # Linear layers for Q, K, V projections for all heads combined
        # We project d_model to d_model, then split into num_heads * d_k
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        # Final linear layer to project concatenated outputs back to d_model
        self.w_o = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(p=dropout_rate)
        self.output_dropout = nn.Dropout(p=dropout_rate)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """Computes scaled dot-product attention for a single head or multiple heads in parallel."""
        # Q, K, V shapes: (batch_size, num_heads, seq_len, d_k)
        # K.transpose(-2, -1) transposes the last two dimensions
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            # Apply mask: fill masked positions with a very small negative number
            # so that their softmax probability becomes zero
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)

        output = torch.matmul(attention_weights, V)
        return output, attention_weights

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask=None) -> torch.Tensor:
        """
        Args:
            query, key, value: Input tensors (batch_size, seq_len, d_model)
            mask: Optional mask tensor (batch_size, 1, 1, seq_len) or (batch_size, 1, seq_len, seq_len)
        Returns:
            Output tensor after multi-head attention (batch_size, seq_len, d_model)
        """
        batch_size = query.size(0)

        # 1. Linear projections and reshape for multiple heads
        # (batch_size, seq_len, d_model) -> (batch_size, seq_len, d_model) ->
        # (batch_size, seq_len, num_heads, d_k) -> (batch_size, num_heads, seq_len, d_k)
        Q = self.w_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 2. Apply scaled dot-product attention for each head in parallel
        x, attn_weights = self.scaled_dot_product_attention(Q, K, V, mask)

        # 3. Concatenate outputs from all heads
        # (batch_size, num_heads, seq_len, d_k) -> (batch_size, seq_len, num_heads, d_k) ->
        # (batch_size, seq_len, d_model)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # 4. Final linear projection
        output = self.w_o(x)
        return self.output_dropout(output)


# --- Demonstration --- 

# Hyperparameters
d_model = 512       # Embedding dimension
num_heads = 8       # Number of attention heads
seq_len = 100       # Sequence length
batch_size = 2      # Batch size
vocab_size = 10000  # Example vocabulary size

print(f"Demonstrating Multi-Head Attention and Positional Encoding with:")
print(f"  d_model={d_model}, num_heads={num_heads}, seq_len={seq_len}, batch_size={batch_size}")

# 1. Create dummy input embeddings
# In a real scenario, this would come from an embedding layer (e.g., nn.Embedding)
# For simplicity, let's assume we have random embeddings.
# Shape: (batch_size, seq_len, d_model)
input_embeddings = torch.randn(batch_size, seq_len, d_model)
print(f"\nInput embeddings shape: {input_embeddings.shape}")

# 2. Apply Positional Encoding
pos_encoder = PositionalEncoding(d_model, max_len=seq_len, dropout_rate=0.1)
embeddings_with_pos = pos_encoder(input_embeddings)
print(f"Embeddings with Positional Encoding shape: {embeddings_with_pos.shape}")

# Verify that positional encoding has been added (values should change)
# Note: Due to dropout, direct comparison might not be exactly equal, but values should be different
print(f"First token embedding before PE (first 5 dims): {input_embeddings[0, 0, :5].tolist()}")
print(f"First token embedding after PE (first 5 dims):  {embeddings_with_pos[0, 0, :5].tolist()}")

# 3. Apply Multi-Head Attention
# For simplicity, we'll use the same embeddings_with_pos for Q, K, V (self-attention)
multi_head_attn = MultiHeadAttention(d_model, num_heads, dropout_rate=0.1)

# Create a dummy mask (e.g., for padding or preventing future tokens from being seen)
# Here, we'll create a mask that allows all tokens to attend to each other (no masking)
# A more realistic mask for decoder self-attention would be lower triangular.
# (batch_size, 1, seq_len, seq_len) for broadcasting
mask = torch.ones(batch_size, 1, seq_len, seq_len, dtype=torch.bool)

output_attn = multi_head_attn(embeddings_with_pos, embeddings_with_pos, embeddings_with_pos, mask=mask)
print(f"Output of Multi-Head Attention shape: {output_attn.shape}")

# Verify that the output is different from the input (contextualized)
print(f"\nFirst token embedding after MHA (first 5 dims): {output_attn[0, 0, :5].tolist()}")

# Example of a causal mask (for decoder self-attention)
# This mask ensures that a token at position 'i' can only attend to tokens at positions <= 'i'
def generate_causal_mask(seq_len):
    mask = (torch.triu(torch.ones(seq_len, seq_len)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask.bool() # Return as boolean for masked_fill

causal_mask = generate_causal_mask(seq_len).unsqueeze(0).unsqueeze(0) # (1, 1, seq_len, seq_len)
print(f"\nCausal mask shape: {causal_mask.shape}")

# Apply MHA with a causal mask
output_attn_causal = multi_head_attn(embeddings_with_pos, embeddings_with_pos, embeddings_with_pos, mask=causal_mask)
print(f"Output of Multi-Head Attention with causal mask shape: {output_attn_causal.shape}")


### Interpreting the Output and Practical Considerations

Let's break down what the code demonstrates and its implications:

#### Interpreting the Code Output

1.  **`Input embeddings shape: torch.Size([2, 100, 512])`**: This represents our batch of input sequences. We have 2 sequences, each 100 tokens long, and each token is represented by a 512-dimensional vector.

2.  **`Embeddings with Positional Encoding shape: torch.Size([2, 100, 512])`**: The shape remains the same, which is crucial. Positional encoding doesn't change the dimensionality; it merely *adds* positional information to the existing token embeddings. You can observe that the numerical values of the embeddings change after applying `PositionalEncoding`, confirming that the positional signal has been injected.

3.  **`Output of Multi-Head Attention shape: torch.Size([2, 100, 512])`**: Again, the output shape is identical to the input shape. This is by design: Multi-Head Attention processes the input and produces a new representation for each token, but it maintains the sequence length and embedding dimension. Each token's output vector now contains information aggregated from all other tokens in the sequence, weighted by their relevance (as determined by the attention mechanism) and from multiple 


perspectives


(heads).

4.  **Causal Masking**: When we apply a `causal_mask`, the attention mechanism is restricted. For a token at position `i`, it can only attend to tokens at positions `j <= i`. This is critical in tasks like language generation, where a model should only predict the next word based on the words it has already generated, not future words. The output shape remains the same, but the internal computations and thus the resulting contextualized embeddings are different.

#### Performance Trade-offs

*   **Computational Cost**: Multi-Head Attention involves multiple linear projections and matrix multiplications. While each head operates on a reduced dimension (`d_k = d_model / num_heads`), the total number of operations for `Q`, `K`, `V` projections and the final output projection scales with `d_model^2`. The attention calculation itself scales with `seq_len^2 * d_k` for each head, summed over `num_heads`. This `seq_len^2` dependency is the primary bottleneck for very long sequences.
*   **Memory Usage**: Storing the `Q`, `K`, `V` matrices for all heads, and especially the attention scores matrix (`batch_size, num_heads, seq_len, seq_len`), can be memory-intensive. This is another factor limiting the maximum sequence length that can be processed on available hardware.
*   **Parallelism**: A significant advantage is that the computations for each attention head are independent and can be performed in parallel. This makes Multi-Head Attention highly efficient on modern GPUs, which excel at parallel matrix operations.

#### Typical Use Cases

Multi-Head Attention and Positional Encoding are the bedrock of the Transformer architecture, enabling its success across a wide range of NLP tasks:

*   **Machine Translation**: Understanding complex grammatical structures and long-range dependencies between words in different languages.
*   **Text Summarization**: Identifying the most salient information in a document by attending to key phrases and sentences.
*   **Question Answering**: Pinpointing the exact answer span in a given context by understanding the relationship between the question and the document.
*   **Language Modeling/Generation**: Predicting the next word in a sequence, where positional encoding ensures correct word order and multi-head attention captures diverse contextual cues.
*   **Code Generation/Completion**: Understanding syntax, variable scope, and logical flow to generate correct and coherent code.

#### Modern Variations (as of 2026)

While the core concepts remain, research continues to refine these components:

*   **Rotary Positional Embeddings (RoPE)**: Instead of adding positional information, RoPE applies a rotation to the query and key vectors based on their absolute position. This method, used in models like LLaMA and Mistral, has shown strong performance, especially for longer contexts, by implicitly encoding relative position information.
*   **Grouped Query Attention (GQA) / Multi-Query Attention (MQA)**: These are optimizations primarily for inference speed in large language models. Instead of having separate `K` and `V` projections for each head, multiple query heads share a single `K` and `V` projection (MQA) or a small group of `K` and `V` projections (GQA). This significantly reduces memory bandwidth requirements during inference, making large models faster and more memory-efficient.

These advancements highlight the ongoing evolution of the Transformer architecture, building upon the fundamental principles of multi-head attention and positional encoding to push the boundaries of what's possible in NLP.


### Resources for Further Learning

*   **The Original Transformer Paper**: "Attention Is All You Need" by Vaswani et al. (2017) - The foundational paper introducing Multi-Head Attention and Positional Encoding.
    *   [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)
*   **PyTorch `torch.nn.MultiheadAttention` Documentation**: PyTorch provides a highly optimized and production-ready implementation of Multi-Head Attention. While our custom implementation helps understand the mechanics, this is what you'd use in real-world applications.
    *   [https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html)
*   **Hugging Face Transformers Library**: Explore how these concepts are integrated into state-of-the-art models like BERT, GPT, and T5.
    *   [https://huggingface.co/docs/transformers/index](https://huggingface.co/docs/transformers/index)
*   **The Illustrated Transformer**: A fantastic visual guide that breaks down the Transformer architecture step-by-step.
    *   [http://jalammar.github.io/illustrated-transformer/](http://jalammar.github.io/illustrated-transformer/)
*   **Google AI Blog - Transformer: A Novel Neural Network Architecture for Language Understanding**: An accessible overview from the creators.
    *   [https://ai.googleblog.com/2017/08/transformer-novel-neural-network.html](https://ai.googleblog.com/2017/08/transformer-novel-neural-network.html)
*   **Rotary Positional Embeddings (RoPE) Paper**: "RoFormer: Enhanced Transformer with Rotary Position Embedding" by Su et al. (2021).
    *   [https://arxiv.org/abs/2104.09864](https://arxiv.org/abs/2104.09864)
*   **Grouped Query Attention (GQA) Paper**: "GQA: Training Generalized Multi-Query Attention for Efficient Transformer Inference" by Ainslie et al. (2023).
    *   [https://arxiv.org/abs/2305.13245](https://arxiv.org/abs/2305.13245)
